In [ ]:
#Importazione librerie
from pathlib import Path
import numpy as np
from skimage import io 
from skimage.transform import resize 
from tqdm import tqdm
import random
import shutil

TRAIN_DIR = Path("../dataset/train")
TEST_DIR = Path("../dataset/test")

TRAIN_RESIZED = Path("../dataset_resized/train_resized")
TEST_RESIZED = Path("../dataset_resized/test_resized")
VALIDATION_RESIZED = Path("../dataset_resized/validation_resized")


In [ ]:
#Controllo perdita di informazione durante la conversione da RGBA a scala di grigi su dataset originale

# Verifica che i canali RGB siano identici e che alpha non contenga informazione
def check_channels(root: Path):
    paths = list(root.rglob("*.png"))

    rgb_different = 0
    alpha_different = 0
    unexpected_shape = 0

    for path in paths:
        image = io.imread(path)

        if image.ndim != 3 or image.shape[2] != 4:
            unexpected_shape += 1
            continue

        if not (
            np.array_equal(image[:, :, 0], image[:, :, 1])
            and np.array_equal(image[:, :, 0], image[:, :, 2])
        ):
            rgb_different += 1

        if not np.all(image[:, :, 3] == 255):
            alpha_different += 1

    print(f"Numero immagini: {len(paths)}")
    print(f"Shape inattese: {unexpected_shape}")
    print(f"RGB diversi: {rgb_different}")
    print(f"Alpha diverso da 255: {alpha_different}")

print("TRAIN")
check_channels(TRAIN_DIR)

print("\nTEST")
check_channels(TEST_DIR)

In [ ]:

#Resize delle immagini a 256x256 e conversione in scala di grigi validata dal controllo precedente
#Salvataggio in dataset_resized/train_resized e dataset_resized/test_resized


def process_images(input_root: Path, output_root: Path, size=(256, 256)):
    image_paths = list(input_root.rglob("*.png"))

    for image_path in tqdm(image_paths, desc=f"Processing {input_root.name}"):

        # Stessa struttura sottocartelle
        rel_path = image_path.relative_to(input_root)
        output_path = output_root / rel_path
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if output_path.exists():
            continue

        # Legge il PNG mantenendo dtype e range originali
        image = io.imread(image_path)

        #cambio di formato, aveva anche canali RGBA, deve diventare solo 256x256 in scala di grigi
        #possibile data la validazione precedente che R=G=B e A=255, quindi basta prendere un canale
        image = image[:, :, 0]

        # Resize
        image_resized = resize(
            image,
            size,
            anti_aliasing=True,
            preserve_range=True,
        )

        image_resized = np.rint(image_resized).astype(image.dtype)

        io.imsave(output_path, image_resized)


process_images(TRAIN_DIR, TRAIN_RESIZED, size=(256, 256))
process_images(TEST_DIR, TEST_RESIZED, size=(256, 256))

print("Resize completato.")

In [ ]:

#Controllo dimensioni, dtype e range delle immagini nel dataset ridimensionato

# Controlla dimensioni, dtype e range delle immagini
def inspect_dataset(root: Path):
    paths = list(root.rglob("*.png"))

    shapes = set()
    dtypes = set()
    global_min = np.inf
    global_max = -np.inf

    for path in paths:
        image = io.imread(path)

        shapes.add(image.shape)
        dtypes.add(image.dtype)

        global_min = min(global_min, image.min())
        global_max = max(global_max, image.max())

    print(f"Numero immagini: {len(paths)}")
    print(f"Shape: {shapes}")
    print(f"Dtype: {dtypes}")
    print(f"Range: [{global_min}, {global_max}]")


print("TRAIN")
inspect_dataset(TRAIN_RESIZED)

print("\nTEST")
inspect_dataset(TEST_RESIZED)

In [ ]:
#Split dataset tra train e validation, 80% train e 20% validation

SEED = 123
N_VALIDATION = 2

# Crea il validation set spostando due cartelle intere dal training
if not VALIDATION_RESIZED.exists():
    patient_dirs = sorted([p for p in TRAIN_RESIZED.iterdir() if p.is_dir()])

    random.seed(SEED)
    validation_patients = random.sample(patient_dirs, N_VALIDATION)

    VALIDATION_RESIZED.mkdir(parents=True)

    for patient_dir in validation_patients:
        shutil.move(str(patient_dir), VALIDATION_RESIZED / patient_dir.name)

    print("Validation set creato.")
    print("Pazienti validation:", [p.name for p in validation_patients])
else:
    print("Validation set già esistente.")

print("Pazienti train:", len([p for p in TRAIN_RESIZED.iterdir() if p.is_dir()]))
print("Pazienti validation:", len([p for p in VALIDATION_RESIZED.iterdir() if p.is_dir()]))



In [ ]:
#Funzione di normalizzazione in [0,1]

def load_normalized_image(path: Path):
    image = io.imread(path)
    return image.astype(np.float32) / 255.0
